# Semaine 2 : Modélisation des Tâches Temps Réel

## Objectifs pédagogiques
- Comprendre les paramètres temporels d’une tâche temps réel : C, T, D, P, offset, jitter.
- Distinguer les tâches périodiques, sporadiques et apériodiques.
- Modéliser les contraintes de précédence et de ressources.
- Calculer l’hyperpériode d’un ensemble de tâches et évaluer l’utilisation du processeur.

## 1. Modèle d’une tâche temps réel

Une tâche temps réel est caractérisée par un ensemble de **paramètres temporels** qui décrivent son comportement périodique ou sporadique.

### Paramètres principaux

| Paramètre | Symbole | Définition |
|-----------|---------|------------|
| **Temps d’exécution pire cas (WCET) ou capacité** | **C** | Durée maximale d’exécution de la tâche sur le processeur, sans interruption. |
| **Période** | **T** | Intervalle de temps entre deux activations/réveils successives (pour une tâche périodique). |
| **Échéance relative / Delai critique** | **D** | Délai maximal autorisé entre l’activation et la fin d’exécution. |
| **Priorité** | **P** | Niveau de priorité de la tâche (utilisé par l’ordonnanceur). |
| **Offset / réveil 0** | **O/r0** | Délai entre le début du système (t=0) et la première activation de la tâche. |
| **Gigue (jitter)** | **J** | Variation maximale du temps d’activation réel par rapport à l’activation théorique. |

> **Note** : On suppose souvent que D = T (échéance égale à la période) et O = 0 (offset nul) pour simplifier l’analyse. Dans ce cas on dit que la tâche est à échéance sur requête.

**Exemple** : une tâche de contrôle moteur a C = 2 ms, T = 10 ms, D = 10 ms, P = haute priorité.

## 2. Types de tâches

### Tâches périodiques
- Activées à intervalles réguliers (période T constante).
- Exemple : acquisition d’un capteur toutes les 20 ms.

### Tâches sporadiques
- Activées par des événements externes, mais avec une **période minimale** entre deux activations (inter-arrivée minimale).
- Exemple : réception d’un message réseau avec un débit maximal.
- On peut les modéliser comme des tâches périodiques avec T = intervalle minimal entre deux arrivées.

### Tâches apériodiques
- Activées de manière irrégulière, sans contrainte de période minimale.
- Exemple : demande utilisateur ponctuelle.
- Nécessitent un traitement particulier (serveurs apériodiques) pour ne pas perturber les tâches périodiques.

## 3. Contraintes de précédence et de ressources

### Contraintes de précédence
- Une tâche ne peut commencer son exécution que si une autre tâche est terminée.
- Exemple : lire les données d’un capteur avant de calculer la commande.
- Modélisées par un graphe de précédence (DAG).

### Contraintes de ressources
- Les tâches partagent des ressources matérielles ou logicielles (mémoire, périphériques, variables partagées).
- L’accès concurrent doit être contrôlé (sémaphores, mutex) pour éviter les incohérences et l’inversion de priorité.
- Ces contraintes seront traitées en détail dans la semaine 5.

---
## Activité 1 : Calcul de l’hyperpériode et de l’utilisation

Pour un ensemble de tâches périodiques, l’**hyperpériode ou Intervalle d'étude** H est le plus petit commun multiple (PPCM) des périodes de toutes les tâches. L’analyse d’ordonnançabilité se fait sur une hyperpériode car le comportement se répète.

L’**utilisation** U du processeur est la somme des rapports C/T de chaque tâche :  
U = ∑ (C_i / T_i)

Si U > 1, l’ordonnancement est impossible sur un processeur unique.

### Tâches à analyser
Soit trois tâches périodiques :
- Tâche 1 : C = 1, T = 4
- Tâche 2 : C = 2, T = 6
- Tâche 3 : C = 3, T = 12

**Questions :**
1. Calculez l’hyperpériode H.
2. Calculez l’utilisation U.
3. Le système est-il potentiellement ordonnançable ?

Utilisez le code ci-dessous pour vérifier vos calculs.

In [1]:
import math
from functools import reduce

def pgcd(a, b):
    return math.gcd(a, b)

def ppcm(a, b):
    return abs(a * b) // pgcd(a, b)

def hyperperiode(periodes):
    return reduce(ppcm, periodes)

def utilisation(taches):
    # taches est une liste de tuples (C, T)
    return sum(C / T for C, T in taches)

# Définition des tâches (C, T)
taches = [(1, 4), (2, 6), (3, 12)]

periodes = [T for _, T in taches]
H = hyperperiode(periodes)
U = utilisation(taches)

print(f"Hyperpériode H = {H}")
print(f"Utilisation U = {U:.4f}")
if U <= 1:
    print("U <= 1 : l'ordonnancement est possible en théorie.")
else:
    print("U > 1 : ordonnancement impossible sur un seul processeur.")

Hyperpériode H = 12
Utilisation U = 0.8333
U <= 1 : l'ordonnancement est possible en théorie.


---
## Activité 2 : Simulation de plusieurs tâches périodiques

Nous allons simuler l’exécution de deux tâches périodiques sur un processeur unique, avec un ordonnanceur à priorité fixe (Rate Monotonic : priorité plus haute pour période plus courte).

La simulation affiche un chronogramme simplifié pour chaque instant : quelle tâche est exécutée et si une échéance est violée.

Modifiez les paramètres (C et T) pour observer le comportement.

In [8]:
def simuler_rm(taches, duree):
    """
    Simule l'ordonnancement Rate Monotonic de tâches périodiques.
    taches : liste de dictionnaires {'nom': str, 'C': int, 'T': int, 'D': int}
    Retourne une liste de tuples (temps, nom_tache_executee, violations)
    """
    # Initialisation des états
    etat = {}
    for t in taches:
        etat[t['nom']] = {
            'prochaine_activation': 0,
            'prochaine_echeance': t['D'],
            'restant': 0,
            'actif': False,
            'deadline_violee': False
        }
    
    temps = 0
    trace = []
    violations = []
    
    while temps < duree:
        # Activation des tâches au début de leur période
        for t in taches:
            nom = t['nom']
            if temps >= etat[nom]['prochaine_activation']:
                etat[nom]['restant'] = t['C']
                etat[nom]['prochaine_activation'] += t['T']
                etat[nom]['prochaine_echeance'] = temps + t['D']
                etat[nom]['deadline_violee'] = False
        
        # Choix de la tâche à exécuter : priorité plus haute (période plus courte) et prête
        tache_exec = None
        for t in sorted(taches, key=lambda x: x['T']):  # tri par période croissante
            nom = t['nom']
            if etat[nom]['restant'] > 0:
                tache_exec = t
                break
        
        if tache_exec:
            nom = tache_exec['nom']
            etat[nom]['restant'] -= 1
            trace.append((temps, nom))
            
            # Vérification des échéances
            for t in taches:
                nom_t = t['nom']
                if temps >= etat[nom_t]['prochaine_echeance'] and etat[nom_t]['restant'] > 0 and not etat[nom_t]['deadline_violee']:
                    violations.append((temps, nom_t))
                    etat[nom_t]['deadline_violee'] = True
        else:
            trace.append((temps, 'IDLE'))
        
        temps += 1
    
    return trace, violations

# Définition de deux tâches
taches = [
    {'nom': 'T2', 'C': 2, 'T': 6, 'D': 6},
    {'nom': 'T3', 'C': 4, 'T': 12, 'D': 12},
    {'nom': 'T1', 'C': 4, 'T': 8, 'D': 8}
]

duree_sim = 24
trace, violations = simuler_rm(taches, duree_sim)

print("Chronogramme (temps : tâche)")
for temps, nom in trace:
    print(f"{temps:2d} : {nom}")

if violations:
    print("\nViolations d'échéance détectées :")
    for temps, nom in violations:
        print(f"  à t={temps}, tâche {nom}")
else:
    print("\nAucune violation d'échéance sur la durée simulée.")

Chronogramme (temps : tâche)
 0 : T2
 1 : T2
 2 : T1
 3 : T1
 4 : T1
 5 : T1
 6 : T2
 7 : T2
 8 : T1
 9 : T1
10 : T1
11 : T1
12 : T2
13 : T2
14 : T3
15 : T3
16 : T1
17 : T1
18 : T2
19 : T2
20 : T1
21 : T1
22 : T3
23 : T3

Aucune violation d'échéance sur la durée simulée.


**Questions :**
1. Essayez avec des valeurs qui rendent l’utilisation U > 1. Que se passe-t-il ?
2. Essayez avec C1=3, T1=5 et C2=3, T2=10. Y a-t-il des violations ? Pourquoi ?
3. Quelle est la limite d’utilisation pour deux tâches selon la condition suffisante de Liu & Layland (RM) ?

---
## Exercice 1 : Analyse de paramètres

Soit une tâche périodique de contrôle de température :
- Le capteur doit être lu toutes les 50 ms (période T).
- Le traitement des données prend au maximum 10 ms (C).
- La commande doit être envoyée au plus tard 20 ms après le début de la période (échéance relative D).
- La première lecture a lieu 5 ms après le démarrage du système (offset O).
- La gigue maximale due au système d’exploitation est de 2 ms (J).

**Questions :**
1. Indiquez les valeurs de T, C, D, O et J.
2. Si D < T, quelle est la conséquence sur l’ordonnancement ?
3. Représentez graphiquement (sur papier ou mentalement) le chronogramme de cette tâche sur deux périodes, en tenant compte de l’offset et de la gigue.

---
## Exercice 2 : Tâches sporadiques

Une tâche sporadique est déclenchée par l’arrivée d’un message sur un bus CAN. Le constructeur garantit que le débit maximal est de 20 messages par seconde.

**Questions :**
1. Quelle est la période minimale T entre deux activations ?
2. Comment modéliser cette tâche sporadique comme une tâche périodique équivalente ?
3. Si le temps d’exécution pire cas C = 5 ms, quelle est l’utilisation de cette tâche ?

---
## TP : Modélisation et simulation avec Python

### Objectif
Écrire un programme Python qui calcule l’hyperpériode et l’utilisation d’un ensemble de tâches, puis simule leur ordonnancement Rate Monotonic sur une hyperpériode.

### Étapes
1. Définissez trois tâches périodiques avec des périodes différentes (par exemple T = 4, 6, 12).
2. Calculez l’hyperpériode et l’utilisation.
3. Implémentez la simulation de l’ordonnancement RM (comme dans l’activité 2) sur la durée de l’hyperpériode.
4. Vérifiez si toutes les échéances sont respectées sur l’hyperpériode.
5. Testez avec un ensemble de tâches dont l’utilisation dépasse 1.
6. (Bonus) Ajoutez une contrainte de précédence : la tâche 2 ne peut commencer que si la tâche 1 est terminée. Adaptez la simulation et observez les changements.

## Références
- Buttazzo, G. – *Hard Real-Time Computing Systems*, Chapitre 2-3.
- Liu, J.W.S. – *Real-Time Systems*, Chapitre 3.